# SciRecQA — Citation-Edge Check on the 100-instance relational subset

**Purpose (plan P0.5):** before spending on the full relational run, verify the relational
gold can be **balanced** — i.e. that the recommended (GT) papers in an instance actually have
**citation links among themselves** (so some asserted relations are *true*, not all hallucinated).

Runs entirely off your Kaggle dataset (the relational checkpoint + `scireqa_full_v1.1.jsonl`).
**No rationale regeneration.** Uses Semantic Scholar (S2AG) to look up references. CPU only;
needs Internet + your `SEMANTIC_SCHOLAR_API_KEY` secret.

Decision: if a healthy fraction of instances have >=1 citation edge (or strong reference
overlap) among their papers -> gold can be balanced -> **proceed to full run**. If GT papers
are citation-disconnected -> enrich instance construction *before* spending.

In [ ]:
# == 1. Imports + locate dataset files + S2AG key =============================
import json, os, re, sys, time, requests
from pathlib import Path
from collections import Counter
import numpy as np
from tqdm import tqdm

def _find(name):
    for root in ["/kaggle/input", "."]:
        r = Path(root)
        if r.exists():
            hits = list(r.rglob(name))
            if hits: return hits[0]
    return None

CK_PATH   = _find("rationales_checkpoint_relational_v1_1.json")
FULL_PATH = (_find("scireqa_full_v1.1.jsonl") or _find("scireqa_full_v1.1.FINAL.jsonl")
             or _find("scireqa_full_v1.1.FINAL2_youden.jsonl"))  # any has identical GT papers
assert CK_PATH and FULL_PATH, "Attach the dataset containing the relational checkpoint + scireqa_full_v1.1.jsonl"

S2_KEY = os.getenv("SEMANTIC_SCHOLAR_API_KEY", "")
try:
    from kaggle_secrets import UserSecretsClient
    S2_KEY = UserSecretsClient().get_secret("SEMANTIC_SCHOLAR_API_KEY") or S2_KEY
except Exception:
    pass
S2_BASE = "https://api.semanticscholar.org/graph/v1"
HDRS    = {"x-api-key": S2_KEY} if S2_KEY else {}
print("checkpoint :", CK_PATH)
print("instances  :", FULL_PATH)
print("S2 API key :", bool(S2_KEY), "(without a key S2AG is heavily rate-limited)")

In [ ]:
# == 2. Join the 100 subset ids with their instances (GT + query papers) ======
sub_ids = set(json.load(open(CK_PATH, encoding="utf-8")).keys())
insts = []
for line in open(FULL_PATH, encoding="utf-8"):
    d = json.loads(line)
    if d["id"] in sub_ids:
        insts.append(d)
    if len(insts) == len(sub_ids):
        break
print(f"matched {len(insts)}/{len(sub_ids)} subset instances")

def s2_id(domain, pid):
    """Map a GT paper_id to an S2AG-queryable id (arXiv ids need the ARXIV: prefix)."""
    pid = str(pid)
    if domain == "arxiv_cs" and re.match(r"^\d{4}\.\d{4,5}", pid):
        return "ARXIV:" + pid
    return pid                      # acl/pubmed paper_id is already an S2 paperId

per_inst, all_papers = [], set()
for d in insts:
    dom = d["domain"]
    gt = [s2_id(dom, p["paper_id"]) for p in d.get("ground_truth_papers", []) if p.get("paper_id")]
    q  = s2_id(dom, d["query_paper_id"]) if d.get("query_paper_id") else None
    per_inst.append({"id": d["id"], "domain": dom, "papers": gt + ([q] if q else [])})
    all_papers.update(gt)
    if q: all_papers.add(q)
print(f"unique papers to resolve via S2AG: {len(all_papers)}")

In [ ]:
# == 3. Resolve each paper -> its S2 paperId + referenced paperIds ============
# One call per paper: fields=paperId,references.paperId. Cached + rate-limited.
# (References list may be capped by S2AG -> edge counts are a CONSERVATIVE lower bound.)
cache = {}
def resolve(pid):
    if pid in cache:
        return cache[pid]
    out = None
    for attempt in range(4):
        try:
            r = requests.get(f"{S2_BASE}/paper/{pid}",
                             headers=HDRS, params={"fields": "paperId,references.paperId"}, timeout=15)
            if r.status_code == 200:
                j = r.json()
                refs = {x.get("paperId") for x in (j.get("references") or []) if x.get("paperId")}
                out = {"pid": j.get("paperId"), "refs": refs}; break
            elif r.status_code == 429:
                time.sleep(2 * (attempt + 1))
            elif r.status_code == 404:
                out = {"pid": None, "refs": set()}; break
            else:
                time.sleep(1)
        except Exception:
            time.sleep(1)
    cache[pid] = out or {"pid": None, "refs": set()}
    time.sleep(0.35)                 # respect S2AG rate limit
    return cache[pid]

for p in tqdm(sorted(all_papers), desc="S2AG resolve"):
    resolve(p)
resolved = sum(1 for v in cache.values() if v["pid"])
print(f"\nresolved {resolved}/{len(cache)} papers ({100*resolved/max(1,len(cache)):.0f}%) "
      f"-> unresolved papers cannot be graph-checked")

In [ ]:
# == 4. Citation edges + reference overlap WITHIN each instance ===============
inst_edge = inst_overlap = cov_ok = 0
edge_counts = []
for rec in per_inst:
    nodes = [(p, cache[p]) for p in rec["papers"] if cache.get(p) and cache[p]["pid"]]
    if len(nodes) >= 2:
        cov_ok += 1
    pid_of  = {p: c["pid"]  for p, c in nodes}
    refs_of = {p: c["refs"] for p, c in nodes}
    P = [p for p, _ in nodes]
    edges = 0
    for i in range(len(P)):
        for j in range(len(P)):
            if i != j and pid_of[P[j]] in refs_of[P[i]]:
                edges += 1                                  # P[i] cites P[j]  (a TRUE builds_on/uses edge)
    overlap = 0
    for i in range(len(P)):
        for j in range(i + 1, len(P)):
            a, b = refs_of[P[i]], refs_of[P[j]]
            if a and b and len(a & b) >= 3:
                overlap += 1                                # >=3 shared references -> genuinely related
    edge_counts.append(edges)
    if edges   >= 1: inst_edge   += 1
    if overlap >= 1: inst_overlap += 1

N = len(per_inst)
print("=== Citation-edge feasibility (100-instance relational subset) ===")
print(f"instances with >=2 graph-resolvable papers (CiteCheck coverage) : {cov_ok}/{N} ({100*cov_ok/N:.0f}%)")
print(f"instances with >=1 DIRECT citation edge among their papers      : {inst_edge}/{N} ({100*inst_edge/N:.0f}%)")
print(f"instances with >=1 reference-overlap pair (>=3 shared refs)     : {inst_overlap}/{N} ({100*inst_overlap/N:.0f}%)")
print(f"avg direct citation edges per instance                          : {np.mean(edge_counts):.2f}")
print(f"instances with EITHER a citation edge or reference overlap      : "
      f"{sum(1 for k in range(N) if edge_counts[k] >= 1 or False)}+ (see overlap above)")

In [ ]:
# == 6. ACL / PubMed edge check (samples instances directly; no relational ckpt) ==
# The relational subset is all arXiv. This checks whether ACL/PubMed GT papers also have
# citation structure, so you can decide whether to include them in the (paid) relational run.
import random
random.seed(0)
N_PER = 50                                   # instances sampled per domain

pool = {"acl_anthology": [], "pubmed": []}
for line in open(FULL_PATH, encoding="utf-8"):
    d = json.loads(line)
    if d["domain"] in pool and len(pool[d["domain"]]) < N_PER * 4:
        pool[d["domain"]].append(d)
dom_inst = {}
for dom, lst in pool.items():
    random.shuffle(lst)
    recs = []
    for d in lst[:N_PER]:
        gt = [s2_id(dom, p["paper_id"]) for p in d.get("ground_truth_papers", []) if p.get("paper_id")]
        q  = s2_id(dom, d["query_paper_id"]) if d.get("query_paper_id") else None
        recs.append({"papers": gt + ([q] if q else [])})
    dom_inst[dom] = recs

new_papers = sorted({p for recs in dom_inst.values() for r in recs for p in r["papers"] if p not in cache})
print(f"resolving {len(new_papers)} new ACL/PubMed papers via S2AG...")
for p in tqdm(new_papers, desc="S2AG ACL/PubMed"):
    resolve(p)

def edge_report(recs, tag):
    ie = io = cov = 0; ec = []
    for rec in recs:
        nodes = [(p, cache[p]) for p in rec["papers"] if cache.get(p) and cache[p]["pid"]]
        if len(nodes) >= 2: cov += 1
        pid = {p: c["pid"] for p, c in nodes}; rf = {p: c["refs"] for p, c in nodes}
        P = [p for p, _ in nodes]; e = 0
        for i in range(len(P)):
            for j in range(len(P)):
                if i != j and pid[P[j]] in rf[P[i]]: e += 1
        ov = 0
        for i in range(len(P)):
            for j in range(i + 1, len(P)):
                if rf[P[i]] and rf[P[j]] and len(rf[P[i]] & rf[P[j]]) >= 3: ov += 1
        ec.append(e)
        if e  >= 1: ie += 1
        if ov >= 1: io += 1
    n = max(1, len(recs))
    print(f"{tag:14s}: resolvable={cov}/{len(recs)} | citation-edge={ie}/{len(recs)} ({100*ie/n:.0f}%) | "
          f"ref-overlap={io}/{len(recs)} ({100*io/n:.0f}%) | avg edges={np.mean(ec):.2f}")

print("\n=== ACL / PubMed citation-edge feasibility ===")
for dom in ["acl_anthology", "pubmed"]:
    edge_report(dom_inst[dom], dom)
print("\nSame rule: healthy edge/overlap (>=30-40%) -> include that domain in the relational run;")
print("very low (esp. if resolvable is low) -> S2AG coverage gap -> scope that domain out or rely on arXiv.")

## 5. Decision

Read the numbers above:

- **Citation edge OR reference-overlap present in a healthy share of instances (say >=30-40%)**
  -> the GT papers are genuinely related; the relational gold will contain **true** relations
  (graph-supported) *and* hallucinated ones (graph-refuted) -> **balanced gold -> proceed to the full run.**
- **Very few edges/overlap (GT papers citation-disconnected)** -> almost every asserted relation
  would be refuted -> CiteCheck's win becomes trivial ("always unsupported") and a reviewer will
  catch it. **Fix instance construction first** (enrich GT sets with citation-linked papers) *before* spending.

Note: edge counts are a **lower bound** (S2AG caps the references list, and arXiv->S2 id resolution
is imperfect). Low `coverage` (resolved %) usually means an id-format issue, not a real absence of links —
inspect a few unresolved ids before concluding.